# M3.2 — Correct the Galaxy10 split (Member 3)

**Goal:** Produce a *candidate* leakage-aware split for Member 2, using the real 17,675-image `Galaxy10_DECals_NoDuplicated.h5` and your completed M3.1 audit.

**Run this in the SAME Kaggle notebook/session that has the 2.73 GB dataset in `/kaggle/working/galaxy_data`.** A separate Kaggle notebook does not automatically share `/kaggle/working` files. If you open this as a new notebook, attach the Galaxy10 dataset or download it again.

Upload/attach `member3_m32_package.zip` using **Input → Upload → New Dataset**, then **Add Input**. You do not need to upload the 2.73 GB file if it is still present in the current session.

This notebook does **not** change your HDF5 or the training code. It groups all galaxies that *might* overlap geometrically at 256px using a conservative sky-circle bound. This can group separate nearby galaxies, so the split requires team approval. Your original three visually suspicious pairs will be verified against the candidate split.

## Step 1 — Locate and unpack the M3.2 package

In [ ]:
import os, sys, zipfile
from pathlib import Path
WORK = Path('/kaggle/working/member3_m32_work')
WORK.mkdir(parents=True, exist_ok=True)

package_matches = [p for base in ['/kaggle/input', '/kaggle/working']
                   for p in Path(base).rglob('member3_m32_package.zip')]
package_matches = [p for p in package_matches if p.is_file()]
if not package_matches:
    raise FileNotFoundError('Attach member3_m32_package.zip via Kaggle Add Input first.')
PACKAGE = package_matches[0]
print('Package:', PACKAGE)
with zipfile.ZipFile(PACKAGE) as z:
    z.extractall(WORK)
sys.path.insert(0, str(WORK))
print('Files:', sorted(p.name for p in WORK.iterdir()))

## Step 2 — Run synthetic unit tests (not research results)

In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'unittest', 'test_member3_m32', '-v'], cwd=str(WORK), check=True)
print('PASS: M3.2 unit tests')

## Step 3 — Find the actual Galaxy10 file and recover your original audit

In [ ]:
from pathlib import Path
# Prior audit is bundled in the package you uploaded to ChatGPT.
AUDIT_ZIP = WORK / 'member3_audit_results.zip'
assert AUDIT_ZIP.is_file(), 'Missing original audit ZIP in package.'
PRIOR = WORK / 'prior_audit'
PRIOR.mkdir(exist_ok=True)
with zipfile.ZipFile(AUDIT_ZIP) as z:
    z.extractall(PRIOR)
PRIOR_REPORT = PRIOR / 'galaxy10_current_audit.json'
PRIOR_NPZ = PRIOR / 'split_seed16' / 'split_indices.npz'
assert PRIOR_REPORT.exists() and PRIOR_NPZ.exists()

wanted = 'galaxy10_decals_noduplicated.h5'
paths=[]
for base in ['/kaggle/working/galaxy_data','/kaggle/input']:
    if Path(base).exists():
        paths.extend([p for p in Path(base).rglob('*.h5') if p.name.lower() == wanted])
if len(paths) != 1:
    raise RuntimeError(f'Expected one Galaxy10 NoDuplicated H5, found {len(paths)}: {paths}. Attach or download the dataset.')
G10 = paths[0]
print('Galaxy10:', G10)
print('Dataset size (GB):', round(G10.stat().st_size/1e9,3))
print('Prior audit:', PRIOR_REPORT)
print('Prior split:', PRIOR_NPZ)

## Step 4 — Generate the corrected split from the REAL dataset

In [ ]:
from member3_m32 import build_corrected
OUT = Path('/kaggle/working/member3_corrected_split')
summary, verification = build_corrected(str(G10), str(PRIOR_NPZ), str(PRIOR_REPORT), str(OUT), footprint_px=256)
print('Verification:', verification['status'])
print('SHA256:', summary['dataset_sha256'])
print('Sky-overlap candidate pairs:', summary['n_pairs'])
print('Original cross-split overlap edges:', summary['n_original_cross_partition_edges'])
print('Rows reassigned to a different partition:', summary['n_changed_partition'])
print('Corrected split sizes:', verification['sizes'])
print('Output directory:', OUT)

## Step 5 — Check the exact 3 suspicious pairs and split balance

They must now be assigned to the **same** partition. This is *not* proof they are duplicates, just that the corrected split prevents these particular pairs crossing partitions. We also show actual class distribution and warn about the one blank image (index 11475) found in M3.1.

In [ ]:
import numpy as np
from member3_m32 import PARTS
with np.load(OUT/'corrected_split_indices.npz', allow_pickle=False) as obj:
    ix={k:obj[k] for k in obj.files}
part_of={int(i):name for name in PARTS for i in ix[name]}
for a,b in [(17475,17332),(8031,9076),(16062,17231)]:
    print(f'{a} <-> {b}: {part_of[a]} / {part_of[b]}')
    assert part_of[a]==part_of[b]

print('Original partition sizes:', summary['original_sizes'])
print('Corrected partition sizes:', summary['corrected_sizes'])
print('Corrected per-class counts:')
for part, counts in verification['class_counts'].items():
    print(f'{part:10s}', counts)
print('Blank image 11475 partition:',part_of[11475], '(needs review; not removed)')
print('Reminder: external decals_images.h5 is NOT approved or included.')

## Step 6 — Export all files for Member 2

The generated ZIP contains the **candidate** split NPZ, manifest with dataset SHA256, detailed geometry/overlap CSV, validation JSON and Member 2 handoff note. **The trainer must be explicitly updated to load these indices**; saving the NPZ alone does not change the training script. Train A and B from scratch with the same file after team approval.

In [ ]:
import shutil
from IPython.display import FileLink, display
ZIP = shutil.make_archive('/kaggle/working/member3_corrected_split_handoff','zip',str(OUT))
print('Created:',ZIP)
print('Size MB:',round(Path(ZIP).stat().st_size/1e6,2))
print('Files:',[p.name for p in OUT.iterdir()])
display(FileLink(ZIP))

## Step 7 — What to send Member 2

Send `member3_corrected_split_handoff.zip` and these numbers: dataset SHA256, corrected counts, geometric overlap pairs, changed partition rows, and verification status. Ask Member 2 to **integrate exact-index loading and fail on mismatched hash**, then test that no split is regenerated in either run. Both experiments must use the same corrected NPZ.

**Before approval:** inspect `overlap_review.csv`; the algorithm groups possible sky-footprint overlaps, not manually confirmed duplicates. The one constant-value galaxy, external cutouts, image provenance, and any excessively changed class balance still need signoff. A passing test means *the implemented guard held for this dataset*, not that no possible leakage exists anywhere.